In [ ]:
!pip install catboost
!pip install openai
import pandas as pd
import numpy as np
import sqlite3

from catboost import CatBoostRegressor
import google.generativeai as genai

In [ ]:
df = pd.read_csv(r"/content/engineered_housing.csv")

print(df.shape)
df.drop(columns=["Unnamed: 0"], inplace=True)
df.head()

(16209, 32)


,price,bedrooms,bathrooms,sqft_living,sqft_lot,waterfront,view,condition,grade,zipcode,...,location_cluster,zip_sqft_mean,zip_sqft_median,sqft_per_room,sqft_ratio,lat_long_product,age_sqft_interaction,grade_sqft,bath_per_bed,log_price
0,268643,4,2.25,1810,9240,0,0,3,7,98055,...,5,1780.688119,1695.0,249.655172,1.089705,-5796.086969,97740,12670,0.450,12.501139
1,245000,3,2.50,1600,2788,0,0,4,7,98031,...,5,1940.064039,1850.0,246.153846,0.929692,-5792.079236,36800,11200,0.625,12.409013
2,200000,4,2.50,1720,8638,0,0,3,8,98003,...,0,1920.354680,1820.0,229.333333,0.919294,-5781.784435,36120,13760,0.500,12.206073
3,352499,2,2.25,1240,705,0,0,3,7,98027,...,5,2517.959596,2370.0,236.190476,0.999194,-5802.386043,7440,8680,0.750,12.772803
4,232000,3,2.00,1280,13356,0,0,3,7,98042,...,2,2011.756440,1940.0,213.333333,0.804525,-5782.828491,26880,8960,0.500,12.354493


In [ ]:
price_model = CatBoostRegressor()

price_model.load_model("catboost_model.cbm")

print("CatBoost Model Loaded")

CatBoost Model Loaded


In [ ]:
print(df.columns.tolist())

['price', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'waterfront', 'view', 'condition', 'grade', 'zipcode', 'lat', 'long', 'sqft_living15', 'total_sqft', 'has_basement', 'house_age', 'was_renovated', 'years_since_renovation', 'bedrooms_per_sqft', 'relative_living_size', 'floors_cat', 'location_cluster', 'zip_sqft_mean', 'zip_sqft_median', 'sqft_per_room', 'sqft_ratio', 'lat_long_product', 'age_sqft_interaction', 'grade_sqft', 'bath_per_bed', 'log_price']


In [ ]:
feature_cols = [
    col for col in df.columns
    if col not in ["price", "log_price"]
]

print(len(feature_cols))
print(feature_cols)

29
['bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'waterfront', 'view', 'condition', 'grade', 'zipcode', 'lat', 'long', 'sqft_living15', 'total_sqft', 'has_basement', 'house_age', 'was_renovated', 'years_since_renovation', 'bedrooms_per_sqft', 'relative_living_size', 'floors_cat', 'location_cluster', 'zip_sqft_mean', 'zip_sqft_median', 'sqft_per_room', 'sqft_ratio', 'lat_long_product', 'age_sqft_interaction', 'grade_sqft', 'bath_per_bed']


In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key="YOUR_API_KEY"
)

In [ ]:
def call_llm(prompt):

    response = client.chat.completions.create(
        model="deepseek/deepseek-chat-v3",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    return response.choices[0].message.content

In [ ]:
conn = sqlite3.connect("housing.db")

df.to_sql(
    "properties",
    conn,
    if_exists="replace",
    index=False
)

print("Database Created")

Database Created


In [ ]:
columns = df.columns.tolist()

schema = "\n".join(columns)

print(schema)

price
bedrooms
bathrooms
sqft_living
sqft_lot
waterfront
view
condition
grade
zipcode
lat
long
sqft_living15
total_sqft
has_basement
house_age
was_renovated
years_since_renovation
bedrooms_per_sqft
relative_living_size
floors_cat
location_cluster
zip_sqft_mean
zip_sqft_median
sqft_per_room
sqft_ratio
lat_long_product
age_sqft_interaction
grade_sqft
bath_per_bed
log_price


In [ ]:
def generate_sql(question):
      prompt = f"""
      You are an expert SQL analyst.

      Table Name: properties

      Columns:
      {schema}

      Rules:
      1. Generate valid SQLite SQL.
      2. Return ONLY SQL.
      3. No explanations.
      4. No markdown.
      5. Use existing columns only.

      Question:
      {question}
      """
      sql_query = call_llm(prompt)
      sql_query = sql_query.replace("```sql", "")
      sql_query = sql_query.replace("```", "")
      sql_query = sql_query.strip()

      return sql_query

In [ ]:
def run_sql(sql_query):

    return pd.read_sql_query(
        sql_query,
        conn
    )

In [ ]:
import numpy as np

def predict_price(input_df):

    pred_log = price_model.predict(input_df)[0]

    pred_price = np.exp(pred_log)

    return pred_price

In [ ]:
import shap

explainer = shap.TreeExplainer(price_model)

def explain_prediction(input_df):

    pred_price = predict_price(input_df)

    shap_values = explainer.shap_values(input_df)

    shap_df = pd.DataFrame({
        "feature": feature_cols,
        "shap_value": shap_values[0]
    })

    shap_df["abs_shap"] = shap_df["shap_value"].abs()

    shap_df = shap_df.sort_values(
        "abs_shap",
        ascending=False
    )

    top_features = shap_df.head(5)

    top_feature_text = "\n".join(
        [
            f"{row['feature']} : {row['shap_value']:.4f}"
            for _, row in top_features.iterrows()
        ]
    )

    feature_info = input_df.T
    feature_info.columns = ["value"]

    prompt = f"""
    You are a real estate valuation analyst.

    Predicted Price:
    {pred_price:.2f}

    Property Features:
    {feature_info.head(15).to_string()}

    Top SHAP Drivers:
    {top_feature_text}

    Explain:
    1. Why the model predicted this value.
    2. Which factors contributed most.
    3. What this implies about the property.

    ONLY use provided information.
    Do not speculate.
    """

    explanation = call_llm(prompt)

    return {
        "predicted_price": pred_price,
        "top_features": top_features,
        "explanation": explanation
    }

In [ ]:
def simulate_change(
    input_df,
    feature,
    new_value
):

    original_price = predict_price(
        input_df
    )

    modified = input_df.copy()

    modified[feature] = new_value

    new_price = predict_price(
        modified
    )

    impact = new_price - original_price

    return {
        "feature_changed": feature,
        "old_value": input_df[feature].values[0],
        "new_value": new_value,
        "original_price": original_price,
        "new_price": new_price,
        "impact": impact
    }

In [ ]:
def simulate_from_text(question):

    global current_property

    if current_property is None:

        return """
No active property found.

First run a prediction such as:

Predict a waterfront house with
4 bedrooms,
3 bathrooms,
grade 10,
3000 sqft living area
"""

    q = question.lower()

    import re

    if "waterfront" in q:

        return simulate_change(
            current_property,
            "waterfront",
            1
        )

    elif "grade" in q:

        nums = re.findall(r"\d+", q)

        if len(nums) == 0:
            return "Please specify grade."

        return simulate_change(
            current_property,
            "grade",
            int(nums[0])
        )

    elif "sqft" in q:

        nums = re.findall(r"\d+", q)

        if len(nums) == 0:
            return "Please specify sqft."

        return simulate_change(
            current_property,
            "sqft_living",
            int(nums[0])
        )

    else:

        return """
Supported simulations:

What if grade becomes 10?

What if waterfront becomes 1?

What if sqft becomes 3500?
"""

In [ ]:
current_property = None

In [ ]:
import json
import re

def extract_features(question):

    prompt = f"""
    Extract housing features from the user query.

    Return ONLY valid JSON.

    Features:
    bedrooms
    bathrooms
    sqft_living
    waterfront
    view
    condition
    grade

    If a feature is not mentioned,
    DO NOT include it in the JSON.

    User Query:
    {question}
    """

    response = call_llm(prompt)

    # Remove markdown
    response = response.replace("```json", "")
    response = response.replace("```", "")
    response = response.strip()

    # Extract JSON block
    match = re.search(r"\{.*\}", response, re.DOTALL)

    if match:
        response = match.group()

    return json.loads(response)

In [ ]:
def create_property_record(extracted_features):

    base = {}

    # Start with median property
    for col in feature_cols:
        base[col] = df[col].median()

    # Overwrite user supplied values
    for k, v in extracted_features.items():

        if v is not None and k in base:
            base[k] = v

    # =====================================
    # RECOMPUTE ENGINEERED FEATURES
    # =====================================

    sqft_living = base["sqft_living"]
    sqft_lot = base["sqft_lot"]

    bedrooms = base["bedrooms"]
    bathrooms = base["bathrooms"]

    grade = base["grade"]

    # total_sqft
    base["total_sqft"] = sqft_living + sqft_lot

    # bedrooms_per_sqft
    base["bedrooms_per_sqft"] = bedrooms / max(sqft_living, 1)

    # relative_living_size
    base["relative_living_size"] = (
        sqft_living /
        max(base["sqft_living15"], 1)
    )

    # sqft_per_room
    total_rooms = bedrooms + bathrooms

    base["sqft_per_room"] = (
        sqft_living /
        max(total_rooms, 1)
    )

    # sqft_ratio
    base["sqft_ratio"] = (
        sqft_living /
        max(sqft_lot, 1)
    )

    # lat_long_product
    base["lat_long_product"] = (
        base["lat"] *
        base["long"]
    )

    # grade_sqft
    base["grade_sqft"] = (
        grade *
        sqft_living
    )

    # bath_per_bed
    base["bath_per_bed"] = (
        bathrooms /
        max(bedrooms, 1)
    )

    # age_sqft_interaction
    base["age_sqft_interaction"] = (
        base["house_age"] *
        sqft_living
    )

    property_df = pd.DataFrame([base])

    return property_df

In [ ]:
def predict_from_text(question):

    global current_property

    features = extract_features(question)

    property_df = create_property_record(
        features
    )

    current_property = property_df.copy()

    prediction = explain_prediction(
        property_df
    )

    return {
        "extracted_features": features,
        "prediction": prediction
    }

In [ ]:
def analytics_summary(question):

    sql_query = generate_sql(question)

    result = run_sql(sql_query)

    prompt = f"""
    You are a housing market analyst.

    Question:
    {question}

    SQL Query:
    {sql_query}

    Result:
    {result.head(20).to_string()}

    Provide:

    1. Key Finding
    2. Data-backed Insight
    3. Recommendation

    Only use the data provided.
    """

    explanation = call_llm(prompt)

    return {
        "sql_query": sql_query,
        "result": result,
        "analysis": explanation
    }

In [ ]:
def ask_copilot(question):

    q = question.lower()

    print("\n" + "="*70)
    print("HOUSING ANALYTICS & VALUATION COPILOT")
    print("="*70)

    print(f"\nUser Query:\n{question}\n")

    # ==================================
    # SIMULATION
    # ==================================

    if "what if" in q:

        print("Intent: Simulation\n")

        result = simulate_from_text(question)

        print(result)

        return result

    # ==================================
    # PREDICTION
    # ==================================

    elif (
        "predict" in q
        or "estimate" in q
        or "value" in q
        or "price of" in q
    ):

        print("Intent: Prediction\n")

        result = predict_from_text(question)

        prediction = result["prediction"]

        print(
            f"Predicted Price: ₹{prediction['predicted_price']:,.0f}\n"
        )

        print("Top Drivers:\n")

        print(
            prediction["top_features"][
                ["feature", "shap_value"]
            ]
        )

        print("\nExplanation:\n")

        print(
            prediction["explanation"]
        )

        return result

    # ==================================
    # ANALYTICS
    # ==================================

    else:

        print("Intent: Analytics\n")

        result = analytics_summary(question)

        print("Generated SQL:\n")

        print(result["sql_query"])

        print("\nAnalysis:\n")

        print(result["analysis"])

        return result

In [ ]:
ask_copilot(
"""
Predict a waterfront house
with 4 bedrooms,
3 bathrooms,
grade 10,
3000 sqft living area
"""
)


HOUSING ANALYTICS & VALUATION COPILOT

User Query:

Predict a waterfront house
with 4 bedrooms,
3 bathrooms,
grade 10,
3000 sqft living area


Intent: Prediction

Predicted Price: ₹853,746

Top Drivers:

        feature  shap_value
4    waterfront    0.347306
9           lat    0.187358
7         grade    0.165163
2   sqft_living    0.063602
27   grade_sqft   -0.052555

Explanation:

1. **Why the model predicted this value ($853,746.25):**  
   The predicted price is based on the property's features and their contributions as quantified by the model. Key drivers like **waterfront (0.3473)**, **latitude (0.1874)**, and **high grade (0.1652)** positively increased the value, while **sqft_living (0.0636)** had a smaller positive impact. The negative contribution from **grade_sqft (-0.0526)** slightly offset the price.

2. **Top contributing factors:**  
   - **Waterfront (True)**: The largest positive driver (0.3473), indicating proximity to water significantly boosted value.  
   - **La

{'extracted_features': {'bedrooms': 4,
  'bathrooms': 3,
  'sqft_living': 3000,
  'waterfront': True,
  'grade': 10},
 'prediction': {'predicted_price': np.float64(853746.2523332258),
  'top_features':         feature  shap_value  abs_shap
  4    waterfront    0.347306  0.347306
  9           lat    0.187358  0.187358
  7         grade    0.165163  0.165163
  2   sqft_living    0.063602  0.063602
  27   grade_sqft   -0.052555  0.052555,
  'explanation': "1. **Why the model predicted this value ($853,746.25):**  \n   The predicted price is based on the property's features and their contributions as quantified by the model. Key drivers like **waterfront (0.3473)**, **latitude (0.1874)**, and **high grade (0.1652)** positively increased the value, while **sqft_living (0.0636)** had a smaller positive impact. The negative contribution from **grade_sqft (-0.0526)** slightly offset the price.\n\n2. **Top contributing factors:**  \n   - **Waterfront (True)**: The largest positive driver (0.34

In [ ]:
ask_copilot("""
Predict a waterfront house
with 4 bedrooms
3 bathrooms
grade 10
3000 sqft living area
""")


HOUSING ANALYTICS & VALUATION COPILOT

User Query:

Predict a waterfront house
with 4 bedrooms
3 bathrooms
grade 10
3000 sqft living area


Intent: Prediction

Predicted Price: ₹1,292,888

Top Drivers:

       feature  shap_value
4   waterfront    0.371925
9          lat    0.188298
7        grade    0.179425
27  grade_sqft    0.121222
12  total_sqft    0.101705

Explanation:

1. **Prediction Rationale**: The model predicted a value of **$1,292,888.36** based on the property's features, with the strongest positive drivers being **waterfront status**, **location (latitude)**, and **high grade (construction quality)**. These premiumfeatures align with the property’s **3,000 sqft living space**, **10-grade rating** (top-tier construction), and **waterfront access**, all of which command higher market values.  

2. **Key Contributing Factors**:  
   - **Waterfront (`0.3719` impact)**: The largest driver, indicating ocean/lake access significantly boosts value.  
   - **Latitude (`0.1883`)

{'extracted_features': {'bedrooms': 4,
  'bathrooms': 3,
  'sqft_living': 3000,
  'waterfront': True,
  'grade': 10},
 'prediction': {'predicted_price': np.float64(1292888.3551300918),
  'top_features':        feature  shap_value  abs_shap
  4   waterfront    0.371925  0.371925
  9          lat    0.188298  0.188298
  7        grade    0.179425  0.179425
  27  grade_sqft    0.121222  0.121222
  12  total_sqft    0.101705  0.101705,
  'explanation': "1. **Prediction Rationale**: The model predicted a value of **$1,292,888.36** based on the property's features, with the strongest positive drivers being **waterfront status**, **location (latitude)**, and **high grade (construction quality)**. These premiumfeatures align with the property’s **3,000 sqft living space**, **10-grade rating** (top-tier construction), and **waterfront access**, all of which command higher market values.  \n\n2. **Key Contributing Factors**:  \n   - **Waterfront (`0.3719` impact)**: The largest driver, indicatin